In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import chromadb

d:\Virtual_Environments\mvp_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Dataset\books_cleaned.csv
df = pd.read_csv(r'..\Dataset\Books.csv')
df.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_with_subtitle
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain


Loading huggingface open-source transformer (One time downloading, no API Key)

In [3]:
model = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3022.59it/s]


Creating embedings (vectorize) for descriptions of books (no API) 

In [4]:
descriptions = df['description'].tolist()

embeddings = model.encode(
    descriptions,
    show_progress_bar=True,
    batch_size=32
)

print(embeddings.shape)  # should be (num_books, 768)

Batches: 100%|██████████| 160/160 [10:37<00:00,  3.98s/it]

(5091, 768)


Creating Database of Embeddings (no API)

In [9]:
client = chromadb.PersistentClient(path=r"D:\Documents\MVP\chroma_db")

collection = client.create_collection(
    name="books",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for matching
)

collection.add(
    ids=[str(isbn) for isbn in df['isbn13'].tolist()],
    embeddings=embeddings.tolist(),
    documents=df['description'].tolist(),
    metadatas=df[['title', 'authors', 'categories', 'average_rating', 'thumbnail']].to_dict('records')
)

Query Search funtion

In [10]:
def search_books(query_text, n_results=5):
    query_embedding = model.encode([query_text]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    
    return results

Query Result (no api)

In [11]:
results = search_books("world politics",10)
print(results['ids'])       # matched isbn13s
print(results['metadatas']) # titles, authors, etc.
print(results['distances']) # similarity scores (lower = more similar, for cosine)

[['9780745628479', '9780143035831', '9780575073616', '9780745635323', '9780226777108', '9780375760525', '9780743284790', '9780199296095', '9780393329292', '9780006551393']]
[[{'average_rating': 3.93, 'categories': 'Philosophy', 'authors': 'Adam Swift', 'thumbnail': 'http://books.google.com/books/content?id=VM3R5IfrqT8C&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'title': "Political Philosophy: A Beginner's Guide for Students and Politicians"}, {'thumbnail': 'http://books.google.com/books/content?id=b-ZvDwAAQBAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'title': 'Democracy Matters', 'categories': 'History', 'authors': 'Cornel West', 'average_rating': 3.94}, {'authors': 'Joe Haldeman', 'average_rating': 3.51, 'thumbnail': 'http://books.google.com/books/content?id=FWH3wAEACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'title': 'Worlds', 'categories': 'Fiction'}, {'thumbnail': 'http://books.google.com/books/content?id=koYetXlCc-IC&printsec=frontcover&img=1&zoom=1&edge